In [1]:
import pandas as pd
import re
import time
import logging
import xml.etree.ElementTree as ET
import plotly.express as px
import requests
import pickle
import form4_common as f4
import time
pd.set_option("display.max_columns", 100)


In [2]:
SEC_USER_AGENT = "yoyoman021.@gmail.com"
SEC_HEADERS = {"User-Agent": SEC_USER_AGENT}

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger(__name__)


In [10]:
df = pd.read_pickle("C:/Users/atsye/Desktop/Python_data/quant_projects/director_trading/data_output/ceo_purchases.pkl")
display(df.shape,df.head(2))

(201432, 22)

,date,date_filed,shares,price_per_share,transaction_value,transaction_code,ticker,company_name,director,position,title_of_security,ownership_form,market,market_cap,source_url,accession,is_10b5_1_plan,footnotes,shares_owned_after,pct_of_holdings,cluster_buy_count,is_cluster_buy
0,2026-07-23,2026-07-23,200.0,184.3300,36866.000,Sell,PSMT,PRICESMART INC,Bahrambeygui Sherry S.,Director,"Common Stock, $0.0001 par value per share",Direct,NASDAQ,5.710894e+09,https://www.sec.gov/Archives/edgar/data/104180...,0001628280-26-049447,False,The reported securities are owned directly or ...,38749.0,0.005135,NaN,NaN
1,2026-07-21,2026-07-23,64730.0,78.3441,5071213.593,Sell,CRWV,"CoreWeave, Inc.",Intrator Michael N,CEO and President,Class A Common Stock,Direct,NASDAQ,3.997939e+10,https://www.sec.gov/Archives/edgar/data/176962...,0001769628-26-000327,True,The reported transaction represents a sale eff...,2545916.0,0.024795,NaN,NaN


## Data fill
Locate the missing date fields and try and fill them in using the source url column

In [11]:
missing_idx = df[df['date'].isna()].index
print(len(missing_idx))

4698


In [14]:
for idx in missing_idx:
    url = df.iloc[idx]['source_url']

    try:
        r = requests.get(url, headers=SEC_HEADERS, timeout=30)
        r.raise_for_status()
    except requests.RequestException as e:
        log.warning(f"  failed to download {url}: {e}")

    root = f4._parse_xml_bytes(r.content, source=url)

    date_el = root.find(".//transactionDate/value")
    txn_date = date_el.text if date_el is not None else None

    df.loc[idx, 'date'] = txn_date
    time.sleep(0.10)


In [22]:
df.to_pickle("C:/Users/atsye/Desktop/Python_data/quant_projects/director_trading/data_output/ceo_purchases.pkl")

## Check distribution of date_filed dates

In [111]:
df['quarter'] = df['date_filed'].dt.to_period('Q')
summary = df['quarter'].value_counts()
summary

quarter
2026Q2    20401
2025Q4    16570
2026Q1    15943
2024Q4    15216
2025Q1    14745
2025Q3    14177
2024Q3    13910
2024Q2    13643
2024Q1    12986
2023Q1    12776
2025Q2    12512
2023Q4    12237
2023Q2    11636
2023Q3    10951
2026Q3     3729
Freq: Q-DEC, Name: count, dtype: int64